# Técnica de Stacking con SVM en entornos aislados y señas continuas

Esto se trata de una reconstrucción e intento de mejora del paper "**Dynamic Strategy for Recognizing the Mexican Sign Language Alphabet: Bringing Static and Dynamic Signs**"

El siguiente paso es poder manejar en entornos aislados letras que se deben de mover, es decir que no pueden ser tratadas como tal con una imagen o que es información insuficiente, ya que poseen movimiento, como la **J** que posee una posición inicial igual a la **I** pero que al moverse asemejandose la figura de una **J** entonces se convierte en una seña completamente distinta.

Lo que haremos aquí es tratar de construir un modelo que pueda leer estos movimientos mediante una deconstrucción de videos con el uso de ventaneo de los videos, es decir, dividiremos un video en segmentos equidistantes para poder analizarlo como si de una imágen se tratara, después poder comparar el movimiento con el mismo punto de referencia que estaba en la imágen anterior pero ahora con el movimiento con n desplazamientos de tiempo.

Se crearán varios modelos, entre ellos:
- Un tipo de modelo SVM con metodología Stacking + Padding (Aplanamiento y corrección)
- Un modelo SVM con metodología Pooling (Estadístico)
- Una arquitectura CNN para comparación por X y Y (Por imágenes)
- Una arquitectura CNN para comparación por keypoint (Por imágenes)

Todos estos experimentos se realizaran con los datos dinámicos solamente, sin embargo, el mejor de los 3 modelos (Contando la mejor versión con búsqueda de parámetros de cada uno) se le volverá a realizar un entrenamiento pero ahora con el abecedario completo.

# ***=== Librerías ===***

In [ ]:
from __future__ import annotations

import json
import os
import random
import re
from dataclasses import dataclass
import pathlib
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import csv
from datetime import datetime
from collections import Counter


import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import mediapipe as mp        
import joblib                 # guardar/cargar modelos
import matplotlib.pyplot as plt
import seaborn as sns       

# ---------- Scikit-learn ----------
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
)


from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    ParameterGrid,
    StratifiedKFold,
    train_test_split,
    cross_validate,
)

try:
    from sklearn.model_selection import StratifiedGroupKFold  
except Exception:
    StratifiedGroupKFold = None

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.linear_model import LinearRegression
from sklearn.inspection import permutation_importance

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score, accuracy_score, ConfusionMatrixDisplay

# ***=== Configuraciones ===***

In [ ]:
DEBUG = True
TMP_DEBUG = True
CREATE_VIDEOS = False
FLATTEN_KEYPOINTS = False
KEYPOINTS_SAVE = False
CREATE_MODELS = True
USE_CSV_KEYPOINTS = True  # Si es True, se cargan los CSV ya generados
CHANGE_DIR_NAMES = False

ROOTS = [
    Path(r"DB_LSM\MSL-dynamic-signs\train"),
    Path(r"DB_LSM\MSL-dynamic-signs\test"),
    
]

ROOT_TRAIN = Path(r"DB_LSM\MSL-dynamic-signs\train")
ROOT_TEST = Path(r"DB_LSM\MSL-dynamic-signs\test")


EXTS = {"mp4", "mov", "avi"}
EXTS_EDA = {".mp4", ".mov", ".avi"}

mp_hands = mp.solutions.hands

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ***=== Preprocesamiento ===***

## Objeto para manejo de videos

In [ ]:
@dataclass
class VideoMeta:
    person_id: int
    label: str
    view: str
    take_id: int
    src_path: Path

## Manejo de carpetas

### Expresión regular del formato de carpetas

In [ ]:
RX_MAIN = re.compile(
    r"^S(?P<person>\d+)-(?P<label>[A-Za-z0-9ÑñÁÉÍÓÚÜáéíóúü]+)-(?P<view>[A-Za-z0-9_-]+)-(?P<take>\d+)\.(?:mp4|mov|avi)$",
    re.IGNORECASE,
)

TOKEN_FAIL = object()

*Un pequeño spoiler: Cuando se hagan el manejo de carpetas, 45 carpetas no estarán disponibles porque el ultimo guión está incorrecto, sino que hay un punto (.), entonces faltarán y se debe de hacer un renombrado*

### Función de búsqueda con uso de expresión regular

In [ ]:
def parse_regex(p: Path):
    m = RX_MAIN.match(p.name)
    if not m:
        return None
    return VideoMeta(
        person_id=int(m["person"]),
        label=m["label"].upper(),
        view=m["view"].lower(),
        take_id=int(m["take"]),
        src_path=p,
    )

### Función de búsqueda con uso de Tokens

In [ ]:
def parse_tokens(p: Path):
    stem = p.stem
    parts = stem.split("-")
    if len(parts) != 4:
        return TOKEN_FAIL
    p_raw, label, view, take = parts
    if not p_raw.lower().startswith("s"):
        return TOKEN_FAIL
    try:
        person_id = int(p_raw[1:])
        take_id = int(take)
    except ValueError:
        return TOKEN_FAIL
    return VideoMeta(
        person_id=person_id,
        label=label.upper(),
        view=view.lower(),
        take_id=take_id,
        src_path=p,
    )

*Si por casualidad del destino, las expresiones regulares no funcionaran, una manera de hacer la búsqueda un poco más robusta fue hacerlo con tokens*

### Unión Regex + Tokens

In [ ]:
def parse_filename(p: Path):
    meta = parse_regex(p)
    if not meta:  # si regex no jaló, intenta tokens
        meta = parse_tokens(p)
        if meta is TOKEN_FAIL:
            return None
    return meta

### Exploración de videos

In [ ]:
def discover(root: Path):
    valid = []
    invalid = []
    for vid in root.rglob("*.mp4"):
        meta = parse_filename(vid)
        if meta:
            valid.append(meta)
        else:
            invalid.append(vid)
    return valid, invalid

In [ ]:
if __name__ == "__main__":
    ROOT = Path(r"DB_LSM\MSL-dynamic-signs\train")
    if not ROOT.exists():
        print(f"❌ No existe: {ROOT.resolve()}")

    vids, bads = discover(ROOT)

    print(f"\n❌ Videos no válidos: {len(bads)}")
    print(f"✅ Videos válidos: {len(vids)}")

    for b in bads[:5]:
        print("No válido:", b.name)
    for v in vids[:5]:
        print("Válido:", v)

----------

En la primera iteración, antes de realizar cualquier cambio, habrá aproximadamente 45 erroneos, eso quiere decir que podemos aprovecharnos, cambiar el nombre y asegurarnos que sí se puedan utilizar, y cambiaremos el nombre para no perder esos datos.

-----------

### Expresión regular para incorrectas

In [ ]:
RX_BAD = re.compile(
    r"^(S\d+)-([^.]+)\.([0-9]+)\.(mp4|mov|avi)$",
    re.IGNORECASE
)

### Función de cambio de nombre

In [ ]:
def propose_new_name(name: str):

    if RX_MAIN.match(name):
        return None

    m = RX_BAD.match(name)
    if not m:
        return None

    sN, label_view, take, ext = m.groups()
    # solo cambiamos el separador '.' -> '-'
    new_name = f"{sN}-{label_view}-{take}.{ext.lower()}"
    return new_name

### Función completa de cambios de nombres

In [ ]:
def normalize_dirs(roots):
    changes = []
    skipped_exists = 0
    scanned = 0
    for root in roots:
        if not root.exists():
            print(f"⚠️  No existe: {root.resolve()}")
            continue

        for fp in root.rglob("*"):
            if not fp.is_file():
                continue
            if fp.suffix.lower().lstrip(".") not in EXTS:
                continue

            scanned += 1
            new_name = propose_new_name(fp.name)
            if not new_name:
                continue  

            dst = fp.with_name(new_name)
            if dst.exists():
                skipped_exists += 1
                print(f"Destino ya existe, se omite: {dst.name}")
                continue

            changes.append((fp, dst))

    print(f"\nArchivos escaneados: {scanned}")
    print(f"Renombramientos propuestos: {len(changes)}")
    print(f"Saltados por colisión: {skipped_exists}")

    logs_dir = Path(r"Dinamicos_Aislado\Logs")
    logs_dir.mkdir(parents=True, exist_ok=True)

    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    log_path = logs_dir / f"rename_log_{ts}.csv"

    with log_path.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["src", "dst"])
        for src, dst in changes:
            w.writerow([str(src), str(dst)])

    print(f"📄 Log guardado en: {log_path.resolve()}")

    # Apply
    if not changes:
        return

    if not CHANGE_DIR_NAMES:
        print("CHANGE_DIR_NAMES = False. \nNo se realizaron cambios. Si todo se ve bien, pon CHANGE_DIR_NAMES = True.")
    else:
        for src, dst in changes:
            src.rename(dst)
        print(f"✅ Renombramientos aplicados: {len(changes)}")

In [ ]:
if __name__ == "__main__":
    normalize_dirs(ROOTS)

Si el renombramiento fue correcto, entonces no se debería de encontrar un renombramiento propuesto y el CSV debería de estar vacío

## EDA (Análisis Exploratorio)

### Conteo de instancias en clases

In [ ]:
if DEBUG:

    data = []
    EXTS_EDA = {".mp4", ".mov", ".avi"}

    for root in ROOTS:
        for p in root.rglob("*"):
            if not p.suffix.lower() in EXTS_EDA:
                continue
            try:
                cap = cv2.VideoCapture(str(p))
                w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
                h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
                frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                fps = cap.get(cv2.CAP_PROP_FPS)
                cap.release()
            except:
                w, h, frames, fps = None, None, None, None

            label = p.parent.name

            data.append([str(p), label, w, h, frames, fps])

    df = pd.DataFrame(data, columns=["path", "label", "width", "height", "frames", "fps"])


In [ ]:
if DEBUG:

    plt.figure(figsize=(12,6))
    ax = sns.countplot(data=df, x="label", order=df["label"].value_counts().index)
    plt.title("Número de videos por clase")
    plt.xticks(rotation=90)

    for p in ax.patches:
        height = p.get_height()
        ax.annotate(f"{height}",             
                    (p.get_x() + p.get_width() / 2., height), 
                    ha='center', va='bottom', fontsize=9, color='black', xytext=(0,3), textcoords='offset points')

    plt.show()

### Conteo individual de clase Train

In [ ]:
if DEBUG:

    data = []
    for p in ROOT_TRAIN.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in EXTS_EDA:
            continue

        m = RX_MAIN.match(p.name)
        if m:
            label = m["label"].upper()
        else:
            label = "UNK"

        try:
            cap = cv2.VideoCapture(str(p))
            w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) #TENER EN CUENTA QUE PUEDEN SER MAS GRANDES; CONTARLOS MANUALMENTE PARA PRECISION ACERTADA
            fps = cap.get(cv2.CAP_PROP_FPS)
            cap.release()
        except Exception:
            w = h = frames = fps = None

        data.append([str(p), label, w, h, frames, fps])

    df = pd.DataFrame(data, columns=["path", "label", "width", "height", "frames", "fps"])
    print("df shape:", df.shape)
    print(df.head())

In [ ]:
if DEBUG:

    plt.figure(figsize=(12,6))
    ax = sns.countplot(data=df, x="label", order=df["label"].value_counts().index)
    plt.title("Número de videos por letra (Train)")
    plt.xticks(rotation=90)
    plt.tight_layout()

    for p in ax.patches:
        height = p.get_height()
        ax.annotate(f"{height}",             
                    (p.get_x() + p.get_width() / 2., height), 
                    ha='center', va='bottom', fontsize=9, color='black', xytext=(0,3), textcoords='offset points')

    plt.show()


### Conteo individual de clase Test

In [ ]:
if DEBUG:

    data = []
    for p in ROOT_TEST.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in EXTS_EDA:
            continue

        m = RX_MAIN.match(p.name)
        if m:
            label = m["label"].upper()
        else:
            label = "UNK"

        try:
            cap = cv2.VideoCapture(str(p))
            w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            fps = cap.get(cv2.CAP_PROP_FPS)
            cap.release()
        except Exception:
            w = h = frames = fps = None

        data.append([str(p), label, w, h, frames, fps])

    df = pd.DataFrame(data, columns=["path", "label", "width", "height", "frames", "fps"])
    print("df shape:", df.shape)
    print(df.head())

In [ ]:
if DEBUG:
    plt.figure(figsize=(12,6))
    ax = sns.countplot(data=df, x="label", order=df["label"].value_counts().index)
    plt.title("Número de videos por letra (Test)")
    plt.xticks(rotation=90)
    plt.tight_layout()

    for p in ax.patches:
        height = p.get_height()
        ax.annotate(f"{height}",             
                    (p.get_x() + p.get_width() / 2., height), 
                    ha='center', va='bottom', fontsize=9, color='black', xytext=(0,3), textcoords='offset points')

    plt.show()


In [ ]:
if DEBUG:

    df["duration_sec"] = df["frames"] / df["fps"]
    plt.figure(figsize=(8,5))
    sns.histplot(df["duration_sec"].dropna(), bins=30, kde=False)
    plt.title("Distribución de duración de videos (segundos)")
    plt.xlabel("Duración (s)")
    plt.ylabel("Frecuencia")
    plt.show()

In [ ]:
if DEBUG:
    print("=== Resolution ===")
    res_counts = df.groupby(["width","height"]).size().reset_index(name="count")
    print(res_counts.head())

# ***=== Modelo Stacking + Padding ===***

A la hora de poder realizar el entrenamiento del SVM con los videos, se tiene que tomar un ventaneo de los videos, para que se pueda hacer el entrenamiento, lo separamos en frames equidistantes para hacer evaluación de las imágenes y que vaya comparando la posición en un stride de tamaño n, en donde dependerá de qué tantos frames se usarán.

Las evaluaciones se realizarán en 30 frames, 10 frames, 5 frames y 2 frames.

## Extracción y Aplanamiento de Keypoints

#### Configuración para extracción de Keypoints

In [ ]:
data = []
for p in ROOT_TRAIN.rglob("*"):
    if not p.is_file():
          continue
    if p.suffix.lower().lstrip(".") not in EXTS:
        continue

    m = RX_MAIN.match(p.name)
    if m:
        label = m["label"].upper()
    else:
        label = "UNK"

    try:
        cap = cv2.VideoCapture(str(p))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        cap.release()
    except Exception:
        w = h = frames = fps = None

    data.append([str(p), label, w, h, frames, fps])

df_train = pd.DataFrame(data, columns=["path", "label", "width", "height", "frames", "fps"])
print("df shape:", df_train.shape)

if DEBUG:
    print(df_train.head())

### Función de Extracción de keypoints

In [ ]:
def extractKeypoints(path, num_frames: int):
    cap = cv2.VideoCapture(str(path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        total = 1  # evitar división por cero

    idxs = np.linspace(0, total-1, num_frames)
    idxs = np.clip(np.round(idxs).astype(int), 0, max(0, total-1))

    seq = []
    with mp_hands.Hands(static_image_mode=False, max_num_hands=1) as hands:
        for idx in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, frame = cap.read()
            if not ok or frame is None:
                seq.append([0.0]*63)  # padding si no se pudo leer
                continue

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = hands.process(frame)

            if res.multi_hand_landmarks:
                hand = res.multi_hand_landmarks[0]
                coords = []
                for lm in hand.landmark:
                    coords.extend([lm.x, lm.y, lm.z])
                seq.append(coords)           
            else:
                seq.append([0.0]*63)         # padding si no detecta mano

    cap.release()

    # asegurar longitud exacta
    if len(seq) < num_frames:
        seq += [[0.0]*63] * (num_frames - len(seq))
    elif len(seq) > num_frames:
        seq = seq[:num_frames]

    feat = np.asarray(seq, dtype=float).reshape(num_frames*63)

    if DEBUG:

        print(f"Video: {path}")
        print(f"Frames usados: {len(idxs)}")
        print(f"Dimensión del vector: {feat.shape}")

    return feat

#AFECTA MUCHO EN MEMORIA, PRIMERO EXTRAER FEATURES Y LUEGO SE HACE EL SAMPLEO
# NO INICIALIZAR EL WITHHANDS.

In [ ]:
try:
    print("df_train shape:", df_train.shape)
    print("df_train columns:", list(df_train.columns))
except NameError:
    raise RuntimeError("df_train no existe aún. Corre la celda que lo construye.")

if DEBUG or TMP_DEBUG:
    vec = extractKeypoints(df_train["path"].iloc[0], num_frames=30)
    vec = extractKeypoints(df_train["path"].iloc[0], num_frames=20)
    vec = extractKeypoints(df_train["path"].iloc[0], num_frames=10)
    vec = extractKeypoints(df_train["path"].iloc[0], num_frames=5)
    vec = extractKeypoints(df_train["path"].iloc[0], num_frames=2)

De esta forma podemos ver como sube exponencialmente la cantidad necesaria de keypoints

### Función de Extracción de Keypoints en ambas carpetas

In [ ]:
def extractKeypointsAllDir(roots, num_frames: int = 30):
    X, y, paths = [], [], []
    
    for root in roots:
        videos = list(root.rglob("*.mp4"))
        for video in tqdm(videos, desc=f"Procesando {root}", unit="video"):
            try:
                vec = extractKeypoints(video, num_frames=num_frames)  
                X.append(vec)
                y.append(video.parent.name)   # etiqueta = carpeta padre
                paths.append(str(video))      # ruta completa
            except Exception as e:
                print(f"⚠️ Error con {video}: {e}")

    X = np.asarray(X, dtype=float)

    assert X.ndim == 2 and X.shape[1] == num_frames*63, \
        f"Dimensión inconsistente: {X.shape}"

    return X, np.array(y), np.array(paths)


In [ ]:
def save_features(X, y, paths, num_frames, outdir="Features"):
    os.makedirs(outdir, exist_ok=True)
    base = Path(outdir) / f"keypoints_{num_frames}f"

    df = pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])])
    df.insert(0, "label", np.asarray(y))
    df.insert(0, "path",  np.asarray(paths))
    df.to_csv(str(base) + ".csv", index=False)
    print(f"✅ Guardado CSV: {base}.csv  → {df.shape}")

def load_features(num_frames, outdir=r"Dinamicos_Aislado\Features"):
    base = Path(outdir) / f"keypoints_{num_frames}f.csv"
    df = pd.read_csv(base)
    paths = df.pop("path").values
    y_csv = df.pop("label").values   # opcional
    X = df.values.astype(float)
    return X, y_csv, paths

### Extracción de Keypoints

In [ ]:
"""if DEBUG:
    X, y, paths = extractKeypointsAllDir(ROOTS, num_frames=5)"""

### Procesamiento en Submuestreos

In [ ]:
if FLATTEN_KEYPOINTS:
    print("Extrayendo keypoints con formato a 30 frames...")
    X30, y30, p30 = extractKeypointsAllDir(ROOTS, num_frames=30)

    print("\n\nExtrayendo keypoints con formato a 20 frames...")
    X20, y20, p20 = extractKeypointsAllDir(ROOTS, num_frames=20)

    print("\n\nExtrayendo keypoints con formato a 10 frames...")
    X10, y10, p10 = extractKeypointsAllDir(ROOTS, num_frames=10)

    print("\n\nExtrayendo keypoints con formato a 5 frames...")
    X5,  y5,  p5  = extractKeypointsAllDir(ROOTS, num_frames=5)

    print("\n\nExtrayendo keypoints con formato a 2 frames...")
    X2,  y2,  p2  = extractKeypointsAllDir(ROOTS, num_frames=2)

    print("\n\nExtrayendo keypoints con formato a 1 frames...")
    X1,  y1,  p1  = extractKeypointsAllDir(ROOTS, num_frames=1)

In [ ]:
if KEYPOINTS_SAVE:
    save_features(X30,y30,p30, 30, outdir="Features")
    save_features(X20,y20,p20, 20, outdir="Features")
    save_features(X10,y10,p10, 10, outdir="Features") 
    save_features(X5,y5,p5, 5, outdir="Features") 
    save_features(X2,y2,p2, 2, outdir="Features") 
    save_features(X1,y1,p1, 1, outdir="Features") 

## Entrenamiento de modelo base

In [ ]:
def get_label_from_path(p: str) -> str:
    name = Path(p).name
    m = RX_MAIN.match(name)
    if not m:
        return "UNK"
    return m["label"].upper()

In [ ]:
if USE_CSV_KEYPOINTS and CREATE_MODELS:
    print("Cargando keypoints desde CSV...")
    X30, y30, paths = load_features(num_frames=30, outdir="Features")
    print(f"X shape: {X30.shape}, y shape: {y30.shape}, paths shape: {paths.shape}")

else:
    if CREATE_MODELS:

        print("Extrayendo keypoints con formato a 30 frames...")
        X30, y30, p30 = extractKeypointsAllDir(ROOTS, num_frames=30)

        y30 = np.array([get_label_from_path(p) for p in p30])

        paths_arr = np.asarray(p30)

        is_train = np.array([("train" in str(p).lower()) for p in paths_arr])
        is_test  = np.array([("test"  in str(p).lower()) for p in paths_arr])


        X_train_30, y_train_30 = X30[is_train], np.asarray(y30)[is_train]
        X_test_30,  y_test_30  = X30[is_test],  np.asarray(y30)[is_test]

        print(X_train_30.shape, X_test_30.shape)

In [ ]:
missing = sorted(set(y_test_30) - set(y_train_30))
assert not missing, f"Hay clases en test que no están en train: {missing}"

In [ ]:
le = LabelEncoder()
y_train_enc_30 = le.fit_transform(y_train_30)
y_test_enc_30  = le.transform(y_test_30)

print("Clases:", list(le.classes_))

In [ ]:
if CREATE_MODELS:
    print("Creando modelo con 30 frames...")
    
    baseline = make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=10, gamma="scale", probability=True, class_weight=None)
    )
    baseline.fit(X_train_30, y_train_enc_30)

    y_pred_30 = baseline.predict(X_test_30)


    print("\n=== Baseline ===")
    print("Acc:", accuracy_score(y_test_enc_30, y_pred_30))
    print("F1-macro:", f1_score(y_test_enc_30, y_pred_30, average="macro"))
    print(classification_report(y_test_enc_30, y_pred_30, target_names=le.classes_))

    cm = confusion_matrix(y_test_enc_30, y_pred_30)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm,
                annot=True, fmt="d", cmap="Blues",
                xticklabels=le.classes_,
                yticklabels=le.classes_)
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.title("Matriz de confusión - Test")
    plt.tight_layout()
    plt.show()

In [ ]:
if CREATE_MODELS:
    
    param_grid = {
        "svc__C":     [1, 3, 10, 30, 100],
        "svc__gamma": ["scale", 0.03, 0.1, 0.3]
    }

    pipe = make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", probability=True)
    )

In [ ]:
if CREATE_MODELS:
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gscv = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        verbose=1,
        refit=True
    )
    gscv.fit(X_train_30, y_train_enc_30)

In [ ]:
if CREATE_MODELS:

    print("\n=== GridSearch ===")
    print("Mejor params:", gscv.best_params_)
    print("Mejor F1-macro (CV):", gscv.best_score_)

    best_model = gscv.best_estimator_
    y_pred_gs = best_model.predict(X_test_30)
    print("\n=== Test con mejor modelo ===")
    print("Acc:", accuracy_score(y_test_enc_30, y_pred_gs))
    print("F1-macro:", f1_score(y_test_enc_30, y_pred_gs, average="macro"))

    print(classification_report(y_test_enc_30, y_pred_gs, target_names=le.classes_))

    cm = confusion_matrix(y_test_enc_30, y_pred_gs)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm,
                annot=True, fmt="d", cmap="Blues",
                xticklabels=le.classes_,
                yticklabels=le.classes_)
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.title("Matriz de confusión - Test")
    plt.tight_layout()
    plt.show()


    joblib.dump(best_model, r"Dinamicos_Aislado\modelos\svm_stacking_msl.joblib")
    joblib.dump(le, r"Dinamicos_Aislado\modelos\label_encoder.joblib")
    print("✅ Modelo guardado en models/")

## Entrenamiento de resto de modelos

In [ ]:
def loadXY(X, paths):
    y = np.array([get_label_from_path(p) for p in paths])
    paths_arr = np.asarray(paths)
    paths_norm = np.array([str(p).replace("\\", "/").lower() for p in paths_arr])

    is_train = np.array([("/train/" in s) or s.endswith("/train") for s in paths_norm])
    is_test  = np.array([("/test/"  in s) or s.endswith("/test")  for s in paths_norm])

    X_train, y_train = X[is_train], y[is_train]
    X_test,  y_test  = X[is_test],  y[is_test]

    missing = sorted(set(y_test) - set(y_train))
    assert not missing, f"Hay clases en test que no están en train: {missing}"

    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_test_enc  = le.transform(y_test)
    return X_train, y_train_enc, X_test, y_test_enc, le

In [ ]:
def makeModel(X_train_submuestreo, y_train_submuestreo,
              X_test_submuestreo, y_test_enc_submuestreo,
              le_submuestreo, submuestreo):

    param_grid = {
        "svc__C":     [1, 3, 10, 30, 100],
        "svc__gamma": ["scale", 0.03, 0.1, 0.3],
    }

    pipe = make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", probability=True)
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gscv = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        verbose=1,
        refit=True
    )

    gscv.fit(X_train_submuestreo, y_train_submuestreo)

    print(f"\n=== GridSearch {param_grid['svc__C'], param_grid['svc__gamma']} ===")
    print("Mejor params:", gscv.best_params_)
    print("Mejor F1-macro (CV):", gscv.best_score_)

    best_model = gscv.best_estimator_
    y_pred_gs_submuestreo = best_model.predict(X_test_submuestreo)

    print("\n=== Test con mejor modelo ===")
    print("Acc:", accuracy_score(y_test_enc_submuestreo, y_pred_gs_submuestreo))
    print("F1-macro:", f1_score(y_test_enc_submuestreo, y_pred_gs_submuestreo, average="macro"))
    print(classification_report(y_test_enc_submuestreo, y_pred_gs_submuestreo,
                                target_names=le_submuestreo.classes_))
    
    
    cm = confusion_matrix(y_test_enc_submuestreo, y_pred_gs_submuestreo)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le_submuestreo.classes_,
            yticklabels=le_submuestreo.classes_)
    
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.title("Matriz de confusión - Test")
    plt.tight_layout()
    plt.show()

    outdir = r"Dinamicos_Aislado\modelos"
    os.makedirs(outdir, exist_ok=True)
    model_path = os.path.join(outdir, f"svm_stacking_msl_{submuestreo}.joblib")
    le_path    = os.path.join(outdir, f"label_encoder_{submuestreo}.joblib")

    joblib.dump(best_model, model_path)
    joblib.dump(le_submuestreo, le_path)
    print("✅ Modelo guardado en:", model_path)
    print("✅ LabelEncoder guardado en:", le_path)


In [ ]:
if CREATE_MODELS and FLATTEN_KEYPOINTS:

    print("Creando modelo con 20 frames...")
    X_train_20, y_train_enc_20, X_test_20, y_test_enc_20, le_20 = loadXY(X20, p20)
    
    makeModel(X_train_20, 
              y_train_enc_20, 
              X_test_20, 
              y_test_enc_20, 
              le_20, 
              20)

In [ ]:
if CREATE_MODELS and FLATTEN_KEYPOINTS:

    print("Creando modelo con 10 frames...")
    X_train_10, y_train_enc_10, X_test_10, y_test_enc_10, le_10 = loadXY(X10, y10)

    makeModel(X_train_10, 
          y_train_enc_10,
          X_test_10, 
          y_test_enc_10,
          le_10, 
          10)

In [ ]:
if CREATE_MODELS and FLATTEN_KEYPOINTS:
    
    print("Creando modelo con 5 frames...")
    X_train_5, y_train_enc_5, X_test_5, y_test_enc_5, le_5 = loadXY(X5, y5)

    makeModel(X_train_5, 
            y_train_enc_5,
            X_test_5, 
            y_test_enc_5,
            le_5, 
            5)

In [ ]:
if CREATE_MODELS and FLATTEN_KEYPOINTS:
    print("Creando modelo con 2 frames...")
    X_train_2, y_train_enc_2, X_test_2, y_test_enc_2, le_2 = loadXY(X2, y2)

    makeModel(X_train_2, 
            y_train_enc_2,
            X_test_2, 
            y_test_enc_2,
            le_2, 
            2)

In [ ]:
if CREATE_MODELS and FLATTEN_KEYPOINTS:
    print("Creando modelo con 1 frame...")

    X_train_1, y_train_enc_1, X_test_1, y_test_enc_1, le_1 = loadXY(X1, y1)

    makeModel(X_train_1, 
            y_train_enc_1,
            X_test_1, 
            y_test_enc_1,
            le_1, 
            1)

## Evaluación manual de modelos

In [ ]:
model_path = Path(r"Dinamicos_Aislado\modelos\svm_stacking_msl.joblib") #Dirección de modelo a utilizar
le_path    = Path(r"Dinamicos_Aislado\modelos\label_encoder.joblib")    #Dirección del label encoder
num_frames = 30

if model_path.exists() and le_path.exists():
    model = joblib.load(model_path)
    le    = joblib.load(le_path)
    print("✅ Modelo y LabelEncoder cargados.")
    print("Clases:", list(le.classes_))
else:
    raise FileNotFoundError("No se encontró el modelo o el LabelEncoder.")
X, _, paths = load_features(num_frames)

RX_MAIN = re.compile(
    r"^S(?P<person>\d+)-(?P<label>[A-Za-z0-9ÑñÁÉÍÓÚÜáéíóúü]+)-(?P<view>[A-Za-z0-9_-]+)-(?P<take>\d+)\.(?:mp4|mov|avi)$",
    re.IGNORECASE,
)

TOKEN_FAIL = object()

In [ ]:
paths_arr = np.asarray(paths)
paths_norm = np.array([str(p).replace("\\","/").lower() for p in paths_arr])
is_train = np.array([("/train/" in s) or s.endswith("/train") for s in paths_norm])
is_test  = np.array([("/test/"  in s) or s.endswith("/test")  for s in paths_norm])

X_test  = X[is_test]
p_test  = paths_arr[is_test]

def get_label_from_path(p: str) -> str:
    name = Path(p).name
    m = RX_MAIN.match(name)
    return m["label"].upper() if m else "UNK"

y_test_raw = np.array([get_label_from_path(p) for p in p_test])

missing = sorted(set(y_test_raw) - set(le.classes_))
assert not missing, f"Clases en test no vistas por el encoder: {missing}"

y_test = le.transform(y_test_raw)

y_pred = model.predict(X_test)

print("\n=== Evaluación en TEST ===")
print("Acc:", accuracy_score(y_test, y_pred))
print("F1-macro:", f1_score(y_test, y_pred, average="macro"))
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicción"); plt.ylabel("Real")
plt.title(f"Matriz de confusión - Test ({num_frames}f)")
plt.tight_layout(); plt.show()